# Reproduce `websearch`

Generated from `<inference page: websearch x 1 instance(s)>`. The notebook replays inference + evaluation against the exact config snapshot that produced the original run, so a re-execution should produce a comparable `model_patch` (LLM determinism caveats notwithstanding).

- **Instances:** 1
- **Config id:** `websearch`


## 1. Setup

The notebook's `kernelspec.name = "evomas"` (see metadata at the bottom of the file) tells Jupyter / VSCode to auto-pick the interpreter `setup.ps1` / `setup.sh` registered for `~/.evomas-venv`. As a safety net the first cell also prepends the venv's site-packages to `sys.path` — so even if the kernel falls back to a generic Python 3 (different machine, no `setup.ps1` run), the evomas imports still resolve. Adjust `OLLAMA_BASE_URL` if your Ollama daemon isn't on the default host; `SWEBENCH_API_KEY` is only required by the remote-eval cell at the bottom.

### Picking the kernel in VSCode

If VSCode opens the notebook outside the EvoMas workspace (e.g. straight from `~/Downloads`), it won't auto-resolve the kernelspec and asks you to **Select Kernel**. Two-tier picker:

- **"Python Environments…"** lists raw Python interpreters discovered by the Python extension (system Python, conda envs, `.venv`/`venv` folders inside workspaces). `~/.evomas-venv` is outside the conventional discovery paths, so it does NOT show up here.
- **"Jupyter Kernel…"** lists registered Jupyter kernelspecs (`%APPDATA%\jupyter\kernels\*` on Windows, `~/.local/share/jupyter/kernels/*` on Linux/mac). This is where the EvoMas one lives — pick **"Python 3 (EvoMas)"** here. VSCode remembers the choice per-notebook so you only have to do it once.

If the entry doesn't appear there: `Ctrl+Shift+P` → **"Developer: Reload Window"** so the Jupyter extension re-scans kernelspecs, or run `jupyter kernelspec list` to confirm `evomas` is registered (if not, re-run `setup.ps1` / `setup.sh`).

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

# Defensive sys.path prepend: if the running kernel isn't the
# evomas-venv one (e.g. user opened the notebook on a fresh
# clone without running setup.ps1, or VSCode picked a generic
# Python 3), surface the venv's site-packages so `import
# evomas...` still resolves. Skipped when the active sys
# already points at the venv.
_venv = Path.home() / '.evomas-venv'
if _venv.is_dir() and str(_venv) not in sys.executable:
    for _sp in (_venv / 'Lib' / 'site-packages',
                _venv / 'lib' / 'site-packages'):
        if _sp.is_dir() and str(_sp) not in sys.path:
            sys.path.insert(0, str(_sp))

import evomas.paths  # noqa: F401  # triggers load_dotenv(evomas/.env)
from evomas.core.workflow.runner import run as run_evomas
from evomas.utils.instances import fetch_swebench_instances

# Route Python `logging` records to BOTH the notebook output
# AND a per-run text log so `experiments/generate_report.py`
# can mine handoffs / tool calls / per-LLM-call tokens from
# the same lines the API matrix path writes. `force=True`
# overrides any prior basicConfig (e.g. from a stale kernel)
# so the format actually takes effect.
import logging
RUN_OUTPUT_DIR = Path('notebook-websearch').resolve()
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_OUTPUT_DIR / 'inference.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    force=True,
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
    ],
)
print(f'Mirroring inference logs to {LOG_FILE}')


Mirroring inference logs to C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\inference.log


### Environment variables

All run-time configuration the notebook needs lives in this cell — edit values here rather than chasing them through the code. Each variable falls back to the value the EvoMas .env file / shell already has set (via `os.environ.setdefault`), so the cell is safe to re-run.

- **`OLLAMA_BASE_URL`** — where the Ollama daemon serves. Local default works when `ollama serve` runs on this machine; swap to e.g. `http://192.168.1.50:11434` for a remote host.
- **`SWEBENCH_API_KEY`** — required by the remote-eval cell (section 5) when running `--remote` against sb-cli. Local Docker harness runs don't need it.
- **`EVOMAS_INSTANCES`** — override the SWE-bench instance cache location. The next cell already searches sensible defaults; only set this if your cache is somewhere non-standard.
- **`GOOGLE_API_KEY` / `OPENAI_API_KEY`** — only needed if the inlined config picks a Gemini / OpenAI model instead of Ollama.


In [2]:
# Edit these inline OR export them in your shell before
# launching Jupyter. `setdefault` means values already in the
# environment (e.g. loaded from evomas/.env) win.
os.environ['OLLAMA_BASE_URL'] = 'http://192.168.1.50:11434'
# os.environ.setdefault('SWEBENCH_API_KEY', 'swb_...')
# os.environ.setdefault('EVOMAS_INSTANCES', '/path/to/swebench_instances.jsonl')
# os.environ.setdefault('GOOGLE_API_KEY',   '...')
# os.environ.setdefault('OPENAI_API_KEY',   '...')

# Echo the effective values (mask secrets) so you can verify the cell ran.
for _k in ('OLLAMA_BASE_URL', 'SWEBENCH_API_KEY', 'EVOMAS_INSTANCES',
           'GOOGLE_API_KEY', 'OPENAI_API_KEY'):
    _v = os.environ.get(_k, '')
    if not _v:
        print(f'  {_k:<18} <unset>')
    elif _k.endswith('_API_KEY'):
        print(f'  {_k:<18} {_v[:6]}***({len(_v)} chars)')
    else:
        print(f'  {_k:<18} {_v}')


  OLLAMA_BASE_URL    http://192.168.1.50:11434
  SWEBENCH_API_KEY   swb_QM***(56 chars)
  EVOMAS_INSTANCES   <unset>
  GOOGLE_API_KEY     AIzaSy***(39 chars)
  OPENAI_API_KEY     <unset>


## 2. Inlined config

Exact resolved config the original run used. Tweak hyperparameters here if you want to experiment with variations.

The cell below the config dict renders a mermaid diagram of the topology so you can see the agent-graph shape at a glance. The diagram is regenerated from `CONFIG['edges']` + `CONFIG['agents']` every time the cell runs, so edits to the dict above are reflected immediately.

In [3]:
CONFIG = {   'id': 'websearch',
    'description': 'Single-agent web-research config. The user supplies a free-form question (via '
                   '`--problem` on the CLI or the inline `problem_statement` field on a synthetic '
                   'instance). The agent has one tool — `websearch` — and is instructed to ground '
                   'its answer in the search results, citing sources by URL. No workspace, no file '
                   'tools; the runner uses a throwaway tmpdir as workspace so the chain '
                   'bookkeeping is happy.',
    'entry': 'researcher',
    'end': ['researcher'],
    'edges': [],
    'agents': {   'researcher': {   'class': 'Base agent',
                                    'model': 'ollama/qwen3:8b',
                                    'think': True,
                                    'num_ctx': 8192,
                                    'temperature': 0.2,
                                    'top_p': 0.9,
                                    'num_predict': 2048,
                                    'stop': [],
                                    'max_iters': 6,
                                    'prompts': {   'system': 'You are a research assistant. Answer '
                                                             "the user's question by calling the "
                                                             '`websearch` tool one or more times, '
                                                             'then synthesising the results into a '
                                                             'concise answer that cites the URLs '
                                                             'you used. Hard rules:\n'
                                                             '- Call `websearch` at least once '
                                                             'before answering.\n'
                                                             '- Vary the query if the first hits '
                                                             "aren't relevant.\n"
                                                             "- Quote facts; don't invent ones the "
                                                             "snippets don't support.\n"
                                                             '- Your training data has a knowledge '
                                                             'cutoff. The websearch results '
                                                             'reflect the current state of the web '
                                                             '— TRUST dates and version numbers in '
                                                             'the snippets even if they look '
                                                             'future-dated compared to your '
                                                             'training. Do not second-guess the '
                                                             'search results on the basis of '
                                                             '"that\'s later than I expect".\n'
                                                             '- Final answer format: a short '
                                                             'paragraph (3-6 sentences) followed '
                                                             'by a `Sources:` line listing the '
                                                             'URLs you cited.\n'
                                                             '- If (and only if) the question '
                                                             'explicitly names an output path '
                                                             '(e.g. "write the answer to '
                                                             './answer.md"), call `save_text(path, '
                                                             'content)` with the full final answer '
                                                             "once you've composed it. Preserve "
                                                             'the path the user gave EXACTLY, '
                                                             'including any leading `./` — do not '
                                                             'strip or rewrite it. `content` is '
                                                             'the paragraph + Sources line, '
                                                             'nothing else.\n'
                                                             '- After the final answer (and the '
                                                             'save_text call, if any), emit '
                                                             'nothing else.',
                                                   'user': 'Question:\n{issue_text}'},
                                    'tools': [{'name': 'websearch'}, {'name': 'save_text'}]}}}

In [4]:
from IPython.display import Markdown, display

def _topology_mermaid(cfg):
    """Render the topology as a Mermaid flowchart.

    Mirrors what the topology page's cytoscape canvas shows:
    virtual START/END boundary nodes, one node per agent with
    its class as a second-line label, edges directed left-to-
    right. Renders inline in Jupyter Lab + VSCode Jupyter; if
    the cell falls back to plain text the source stays readable.
    """
    lines = ['graph LR']
    lines.append('    START((START))')
    lines.append('    END((END))')
    for name, block in (cfg.get('agents') or {}).items():
        cls = (block or {}).get('class', '') or ''
        label = f'{name}<br/><i>{cls}</i>' if cls else name
        # Backticks would break the mermaid parser; strip them
        # defensively. Class names never contain them today,
        # this is just future-proofing.
        label = label.replace('`', '')
        lines.append(f'    {name}["{label}"]')
    entry = cfg.get('entry') or ''
    if entry:
        lines.append(f'    START --> {entry}')
    for e in (cfg.get('edges') or []):
        if isinstance(e, dict) and e.get('from') and e.get('to'):
            lines.append(f'    {e["from"]} --> {e["to"]}')
    end_field = cfg.get('end')
    ends = (
        [end_field] if isinstance(end_field, str) and end_field
        else list(end_field or [])
    )
    # Only emit `→ END` for nodes with no outgoing edges (the
    # same wiring rule `graph_builder.py` uses). Hub-in-end
    # nodes with outgoing edges don't get the static edge.
    out_sources = {e.get('from') for e in (cfg.get('edges') or [])
                   if isinstance(e, dict)}
    for n in ends:
        if n and n not in out_sources:
            lines.append(f'    {n} --> END')
    return '\n'.join(lines)

display(Markdown('```mermaid\n' + _topology_mermaid(CONFIG) + '\n```'))


```mermaid
graph LR
    START((START))
    END((END))
    researcher["researcher<br/><i>Base agent</i>"]
    START --> researcher
    researcher --> END
```

## 3. Instances

Self-contained: the notebook regenerates its own instances from zero each run. SWE-bench rows get pulled fresh from HuggingFace (cached under `~/.cache/huggingface`); custom rows are reconstructed from the minimal inputs the user added via the Inference page's `+ Custom` modal.

In [5]:
INSTANCE_IDS = ['websearch-py-latest']


In [6]:
# Pull plan for SWE-bench rows: `{(subset, split): [ids]}`.
# At runtime the cell below calls `fetch_swebench_instances`
# per group and filters down to just these IDs.
SWEBENCH_GROUPS = {}


In [7]:
# Custom-instance inputs (no upstream — added locally via the
# Inference page's `+ Custom` modal). Notebook reconstructs the
# row dict from these fields; nothing else is needed.
CUSTOM_ROWS = [   {   'instance_id': 'websearch-py-latest',
        'repo': '',
        'base_commit': '',
        'problem_statement': 'What is the latest stable Python release? Write the answer to '
                             './answer-py.md',
        'hints_text': '',
        'subset': 'adhoc',
        'split': 'adhoc'}]


In [8]:
# Materialise SWE-bench + custom rows into one JSONL the
# runner consumes. Re-uses `RUN_OUTPUT_DIR` from the setup
# cell so inference.log + prediction JSONL share one folder.
output_dir = RUN_OUTPUT_DIR
output_path = output_dir / 'prediction-websearch.jsonl'
INSTANCES_PATH = output_dir / 'instances.jsonl'

selected = []
for (subset, split), ids in SWEBENCH_GROUPS.items():
    print(f'Fetching {len(ids)} {subset}/{split} row(s) from HuggingFace…')
    selected.extend(fetch_swebench_instances(subset, split, instance_ids=ids))
selected.extend(CUSTOM_ROWS)

with INSTANCES_PATH.open('w', encoding='utf-8') as _fh:
    for _row in selected:
        _fh.write(json.dumps(_row, ensure_ascii=False) + '\n')
print(f'Wrote {len(selected)} instance row(s) -> {INSTANCES_PATH}')

_have = {i['instance_id'] for i in selected}
missing = [iid for iid in INSTANCE_IDS if iid not in _have]
if missing:
    print('Missing rows (id not found in HF or in CUSTOM_ROWS):', missing)
print(f'Ready to run {len(selected)} instance(s).')


Wrote 1 instance row(s) -> C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\instances.jsonl
Ready to run 1 instance(s).


## 4. Inference

Re-runs the EvoMas workflow for each instance with the inlined config. All notebook-produced artefacts (prediction JSONL, evaluation reports, custom-instance sidecar) land under one per-run folder at `notebook-websearch/` so they stay grouped together and don't mix with UI/CLI runs in the repo's `results/` tree.

In [9]:
from evomas.exceptions.errors import OllamaMemoryError

# `output_dir` + `output_path` were created in the instances cell above.
predictions = []
with open(output_path, 'w', encoding='utf-8') as out:
    for inst in selected:
        iid = inst['instance_id']
        print(f'--- {iid} ---')
        try:
            patch = run_evomas(inst, config=CONFIG)
        except OllamaMemoryError as exc:
            print(f'Ollama OOM; aborting: {exc}')
            break
        except Exception as exc:
            print(f'run failed on {iid}: {exc}')
            patch = ''
        rec = {
            'instance_id': iid,
            'model_patch': patch,
            'model_name_or_path': 'evomas-notebook',
        }
        predictions.append(rec)
        out.write(json.dumps(rec) + '\n')
print(f'Wrote {len(predictions)} prediction(s) to {output_path}.')


2026-06-06 11:34:41,835 [WARNING] weave.trace.op: Warning: Traces will not be logged. Call weave.init to log your traces to a project.
 (subsequent messages of this type will be suppressed)


2026-06-06 11:34:41,835 [INFO] evomas.core.workflow.runner: === running websearch-py-latest with inline config (id=websearch) ===


2026-06-06 11:34:41,898 [INFO] evomas.core.workflow.runner: ad-hoc workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\adhoc-websearch-py-latest (no repo/base_commit on instance)


--- websearch-py-latest ---


2026-06-06 11:34:42,074 [INFO] evomas.core.workflow.runner: graph runtime: 1 agents x 2 max revisits => recursion_limit=2


...

2026-06-06 11:36:27,851 [INFO] evomas.agents.researcher: [researcher] no tool calls — stopping loop


2026-06-06 11:36:27,872 [WARNING] evomas.utils.patch: git diff failed: warning: Not a git repository. Use --no-index to compare two paths outside a working tree
usage: git diff --no-index [<options>] <path> <path>

Diff output format options
    -p, --patch           generate patch
    -s, --no-patch        suppress diff output
    -u                    generate patch
    -U, --unified[=<n>]   generate diffs with <n> lines context
    -W, --[no-]function-context
                          generate diffs with <n> lines context
    --raw                 generate the diff in raw format
    --patch-with-raw      synonym for '-p --raw'
    --patch-with-stat     synonym for '-p --stat'
    --numstat             machine friendly --stat
    --shortstat           output only the last line of --stat
    -X, --dirstat[=<param1>,<param2>...]
                          output the distribution of relative amount of changes for each sub-directory
    --cumulative          synonym for --dirstat=cumulati

2026-06-06 11:36:27,874 [WARNING] evomas.core.workflow.runner: websearch-py-latest produced empty patch (workspace clean AND state['researcher'] not a diff)


2026-06-06 11:36:27,874 [INFO] evomas.core.workflow.runner: === websearch-py-latest done: 0-char patch | tokens in=3938 out=1710 total=5648 ===


Wrote 1 prediction(s) to C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\prediction-websearch.jsonl.


## 5. Evaluation

Runs `scripts/evaluation/websearch_eval.py` — the evaluator chosen at notebook-generation time (the `--evaluator` flag on `evomas notebook`). Forwards through the unified evaluator CLI contract: `--predictions / --instances / --report-dir / --run-id / --model`. Output lands at `<report-dir>/{model}.{run-id}.json` plus per-instance folders under `<report-dir>/logs/run_evaluation/<run-id>/<model>/<instance>/`.

In [10]:
# Evaluator baked at notebook-gen time (--evaluator on `evomas notebook`).
EVALUATOR_STEM = 'websearch_eval'
EVALUATOR_NEEDS_WSL = False

first = selected[0] if selected else None
SUBSET = (first or {}).get('subset', 'lite')
SPLIT  = (first or {}).get('split',  'dev')

# All eval artifacts land under `output_dir` (alongside
# instances.jsonl + prediction-*.jsonl).
eval_report_dir = output_dir

import platform
from evomas.paths import BASE_DIR as _BASE_DIR
_script = _BASE_DIR / 'scripts' / 'evaluation' / f'{EVALUATOR_STEM}.py'
if EVALUATOR_NEEDS_WSL and platform.system() == 'Windows':
    from evomas.utils.paths import to_wsl
    cmd = [
        'wsl', '--', 'python3', to_wsl(str(_script)),
        '--predictions', to_wsl(str(output_path)),
        '--instances',   to_wsl(str(INSTANCES_PATH)),
        '--report-dir',  to_wsl(str(eval_report_dir)),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
else:
    cmd = [
        sys.executable, str(_script),
        '--predictions', str(output_path),
        '--instances',   str(INSTANCES_PATH),
        '--report-dir',  str(eval_report_dir),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
print(f'Evaluating via {EVALUATOR_STEM}.py')
print('+ ' + ' '.join(cmd))

eval_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, encoding='utf-8', errors='replace',
)
assert eval_proc.stdout is not None
for line in eval_proc.stdout:
    print(line, end='')
eval_proc.wait()
print(f'\n[evaluation finished with exit code {eval_proc.returncode}]')

# Surface the per-instance artifacts the evaluator wrote.
logs_root = eval_report_dir / 'logs' / 'run_evaluation'
if logs_root.is_dir():
    print('\nPer-instance artifacts:')
    for inst_dir in sorted(logs_root.rglob('*/')):
        if (inst_dir / 'report.json').is_file():
            print(f'  {inst_dir}')
for summary in sorted(eval_report_dir.glob('*.json')):
    print(f'Summary: {summary}')


Evaluating via websearch_eval.py
+ C:\Users\XF\.evomas-venv\Scripts\python.exe C:\Users\XF\Desktop\TFG\EvoMas\scripts\evaluation\websearch_eval.py --predictions C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\prediction-websearch.jsonl --instances C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\instances.jsonl --report-dir C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch --run-id notebook-adhoc-adhoc --model evomas-notebook
  -- websearch-py-latest --
     RESOLVED  C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\answer-py.md

Resolved 1/1 instances (criterion: patch_present OR answer_file_found).
Summary -> C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\evomas-notebook.notebook-adhoc-adhoc.json

[evaluation finished with exit code 0]

Per-instance artifacts:
  C:\Users\XF\Desktop\TFG\EvoMas\examples\websearch_demo\notebook-websearch\logs\run_evaluation\notebook-adhoc-adhoc\ev